##### Import statements:

In [97]:
import os
import pathlib
import inspect
import functools
import pickle
import numpy as np
import pandas as pd
import json
import socket
import multiprocessing as mp
import functools
import itertools
import torch
import matplotlib.pyplot as plt
from matplotlib import colormaps
import random
import string
from copy import deepcopy

hostname = socket.gethostname()

if 'rc.zi.columbia.edu' in hostname:
    from ws.general import find_df_constants, matches_template, class_def2str
    from ws.simulate_task import load_sim_params, load_task_def, simulate_session
    from ws.miscellaneous_sparseauto import mdl_geometry_pipeline, fmt_ae_metadata, generate_hparams_df, causal_mask
    from ws.plot import plot_iterate_autoencoder_results, plot_ccgps_by_layer, plot_pars_by_layer
    base = os.path.join('/', 'mnt', 'smb', 'locker', 'issa-locker', 'users', 'Dan', 'code', 'ws') 
else:
    from simulation_whiskers.general import find_df_constants, matches_template, class_def2str
    from simulation_whiskers.simulate_task import load_sim_params, load_task_def, simulate_session, causal_mask
    from simulation_whiskers.miscellaneous_sparseauto import mdl_geometry_pipeline, fmt_ae_metadata, generate_hparams_df, causal_mask
    #from simulation_whiskers.plot import plot_iterate_autoencoder_results, plot_autoencoder_geometry
    from simulation_whiskers.plot import plot_iterate_autoencoder_results, plot_ccgps_by_layer, plot_pars_by_layer
    base = os.path.join('C:\\', 'Users', 'danie' 'Documents', 'code_libraries', 'simulation_whiskers')

from analysis_metadata.analysis_metadata import Metadata, increment_dir_name, write_metadata
import time

##### Define parameters:

In [136]:
# Define task definitions:
task_defs = [
    
    # Task 0:
    [
     functools.partial(matches_template, template={'freq_sh' : 2}), 
     functools.partial(matches_template, template={'freq_sh' : 15})
     ],
    
    # Task 1:
    [
     functools.partial(matches_template, template={'time_mov' : 10}),
     functools.partial(matches_template, template={'time_mov' : 17})
     ]
    ]

# Define general variables:
n_files = 10 
#n_files = 2 # < For debugging
n_geo_subsamples = 1
sum_inpt=False
xor=True
zscore_data = False
sig_init = 1.0
save_learning = False
chunked_reconstruction_loss = False

# Define simulation parameters:
concavity = [0]
n_whisk = 2
prob_poiss = 1.01
noise_w = 0.3
spread = 'auto'
speed = 2.0
ini_phase_m = 0
ini_phase_spr = 100
delay_time = 0
freq_m = 3.0
freq_std = 0.1
std_reset = 0
t_total = 2
dt = 0.1
dx = 0.01
n_trials_pre = 200
#n_trials_pre = 50 # < For debugging
amp = 2
freq_sh = [2, 15]
z1 = [4]
max_rad = 50
n_rad = 4
disp = 4.5
theta = [0]
steps_mov = [10, 17]
rad_vec = [6]
init_position = 0

# Entangler model parameters:
mdl_type = 'autoencoder'
n_hidden = 80
beta_rec = 1.0
beta_pr = 0.1
beta_sp = 0
beta0 = 0
beta1 = 0
beta_xor = 0
p_norm = 1
n_epochs = 50
#n_epochs = 50 # < For debugging
batch_size = 64
lr = 0.001
sig_init = 1 
sig_neu = 0.1 

# Compute parameters:
gpu = False
n_cores = 1

# Do some custom, ad-hoc hyperparameter selection:
#beta_prs=10**np.arange(0, 5, 1)
#beta_prs= [0, 10**-6, 10**-1]
#beta_prs = 10**np.flip(-1*np.arange(0, 100, 100/3))
#beta_prs = 10**np.flip(-1*np.arange(0, 2, 2/100)).astype(float)
beta_prs = np.arange(0.01, 1, 0.09/100)
beta_prs = np.array([0] + list(beta_prs))
#beta_prs = [0, 10**2.5, 10**5]
#n_hiddens = [40, 240]
n_hiddens = [80]
hparams = [{'beta_pr':x[0], 'n_hidden':x[1]} for x in list(itertools.product(beta_prs, n_hiddens))]
#hparams = None

# Output directory:
if 'rc.zi.columbia' in hostname:
    base_output_directory = os.path.join(base, 'results')
else:
    base_output_directory='E:\\simulation_whiskers\\results\\'
run_base_name='run'
sv=True

##### Generate dataframe of hyperparameters:

In [112]:
default_hparams = {'task_defs' : task_defs, 'n_files' : n_files, 'xor' : xor, 'n_geo_subsamples' : n_geo_subsamples, 
    'zscore_data' : zscore_data, 'save_perf' : False, 'sum_inpt' : sum_inpt, 'chunked_reconstruction_loss' : False, 
    'save_learning' : save_learning, 'gpu' : gpu, 'save_sessions' : False, 'verbose' : False, 'concavity' : concavity, 
    'n_whisk' : n_whisk, 'prob_poiss' : prob_poiss, 'noise_w' : noise_w, 'spread' : spread, 'speed' : speed, 'ini_phase_m' : ini_phase_m, 
    'ini_phase_spr' : ini_phase_spr, 'delay_time' : delay_time, 'freq_m' : freq_m, 'freq_std' : freq_std, 'std_reset' : std_reset, 
    't_total' : t_total, 'dt' : dt, 'dx' : dx, 'n_trials_pre' : n_trials_pre, 'n_repeats' : n_files, 'amp' : amp, 'freq_sh' : freq_sh, 
    'z1' : z1, 'max_rad' : max_rad, 'n_rad': n_rad, 'disp' : disp, 'theta' : theta, 'steps_mov' : steps_mov, 'rad_vec' : rad_vec, 
    'init_position' : init_position, 'mdl_type' : 'autoencoder', 'n_hidden' : n_hidden, 'sig_init' : sig_init, 'sig_neu' : sig_neu, 
    'lr' : lr, 'beta_rec' : beta_rec, 'beta_sp' : beta_sp, 'beta_pr' : beta_pr, 'n_epochs' : n_epochs, 'batch_size' : batch_size, 
    'p_norm' : p_norm, 'beta0' : beta0, 'beta1' : beta1, 'beta_xor' : beta_xor, 'mdl_type' : mdl_type} 
hparams_df = generate_hparams_df(defaults=default_hparams, hparams_manual=hparams)

# Verify parameters before executing:
hparam_strs = list(hparams_df.apply(lambda x : 'model={}, n_hidden={}, beta_rec={}, beta_sp={}, beta_pr={}, n_epochs={}'.format(x.mdl_type,x.n_hidden, x.beta_rec, x.beta_sp, x.beta_pr, x.n_epochs), axis=1))
print('Running following hyperparameters:\n')
print('\n'.join(hparam_strs))
if '__file__' not in dir():
    yn = input('\nProceed? (y/n)')
    if yn == 'y':
        pass
    else: 
        raise AssertionError('User aborted execution.')

Running following hyperparameters:

model=autoencoder, n_hidden=80, beta_rec=1.0, beta_sp=0, beta_pr=0.0, n_epochs=50
model=autoencoder, n_hidden=80, beta_rec=1.0, beta_sp=0, beta_pr=0.0, n_epochs=50
model=autoencoder, n_hidden=80, beta_rec=1.0, beta_sp=0, beta_pr=0.0, n_epochs=50
model=autoencoder, n_hidden=80, beta_rec=1.0, beta_sp=0, beta_pr=0.0, n_epochs=50
model=autoencoder, n_hidden=80, beta_rec=1.0, beta_sp=0, beta_pr=0.0, n_epochs=50
model=autoencoder, n_hidden=80, beta_rec=1.0, beta_sp=0, beta_pr=0.01096478196143185, n_epochs=50
model=autoencoder, n_hidden=80, beta_rec=1.0, beta_sp=0, beta_pr=0.01096478196143185, n_epochs=50
model=autoencoder, n_hidden=80, beta_rec=1.0, beta_sp=0, beta_pr=0.01096478196143185, n_epochs=50
model=autoencoder, n_hidden=80, beta_rec=1.0, beta_sp=0, beta_pr=0.01096478196143185, n_epochs=50
model=autoencoder, n_hidden=80, beta_rec=1.0, beta_sp=0, beta_pr=0.01096478196143185, n_epochs=50
model=autoencoder, n_hidden=80, beta_rec=1.0, beta_sp=0, beta_pr


Proceed? (y/n) y


##### Simulate whisker data:

In [90]:
# Put simulation params into dict:
sim_params = {
    'n_whisk' : n_whisk,
    'prob_poiss' : prob_poiss,
    'noise_w' : noise_w,
    'spread' : spread,
    'speed' : speed,
    'ini_phase_m' : ini_phase_m,
    'ini_phase_spr' : ini_phase_spr, 
    'delay_time' : delay_time, 
    'freq_m' : freq_m, 
    'freq_std' : freq_std,
    'std_reset' : std_reset,
    't_total' : t_total,
    'dt' : dt,
    'dx' : dx,
    'n_trials_pre' : n_trials_pre, 
    'amp' : amp,
    'freq_sh' : freq_sh,
    'z1' : z1,
    'max_rad' : max_rad,
    'n_rad' : n_rad,
    'disp' : disp,
    'theta' : theta,
    'steps_mov' : steps_mov,
    'rad_vec' : rad_vec,
    'init_position' : init_position,
}

n_feat = n_whisk*2

# Simulate train and test sessions: 
train_session=simulate_session(sim_params, sum_bins=False)
train_session['split'] = 'train'
train_session['trial_num'] = np.arange(train_session.shape[0])

test_session=simulate_session(sim_params, sum_bins=False)
test_session['split'] = 'test'
test_session['trial_num'] = np.arange(test_session.shape[0])

# Merge train and test:
sim_df = pd.concat([train_session, test_session], axis=0)

# Unrwap features:
sim_df['features'] = sim_df.apply(lambda x : np.reshape(x.features,-1), axis=1)
sim_df = sim_df.rename(columns={'features':'predictor_features'})
sim_df['predicted_features'] = sim_df['predictor_features']

##### Iterate over hyperparameters, fit entangler models:

In [91]:
simulation_cols = ['concavity', 'n_whisk', 'prob_poiss', 'noise_w', 'spread',
     'speed', 'ini_phase_m', 'ini_phase_spr', 'delay_time', 'freq_m', 'freq_std',
     'std_reset', 't_total', 'dt', 'dx', 'n_trials_pre', 'n_files', 'amp', 'freq_sh',
     'z1', 'max_rad', 'n_rad', 'disp', 'theta', 'steps_mov', 'rad_vec', 'init_position']

autoencoder_cols = ['mdl_type', 'n_hidden', 'sig_init', 'sig_neu', 'lr', 'beta_rec', 'n_epochs', 'batch_size', 'beta_sp', 'p_norm', 'beta_pr', 
    'beta0', 'beta1', 'beta_xor']

# Iterate over dicts of hyperparamter combos:
geo_results = pd.DataFrame()
perf_results = pd.DataFrame()
ae_results = pd.DataFrame()
start_mdl = time.time()
for hidx, curr_hparams in hparams_df.iterrows():

    # Get current model hypermarameter set:
    curr_autoencoder_params = dict(curr_hparams[autoencoder_cols])
    mdl_id = ''.join(random.choices(string.ascii_letters+string.digits, k=10))
    
    # Fit model, test geometry:
    curr_results=mdl_geometry_pipeline(sim_params,  
        tasks=curr_hparams.task_defs, autoencoder_params=curr_autoencoder_params, xor=curr_hparams.xor, 
        n_geo_subsamples=curr_hparams.n_geo_subsamples, zscore_data=curr_hparams.zscore_data, 
        save_perf=False, sum_inpt=curr_hparams.sum_inpt, chunked_reconstruction_loss=curr_hparams.chunked_reconstruction_loss, 
        sessions_in=sim_df, save_learning=curr_hparams.save_learning, gpu=curr_hparams.gpu, save_sessions=False, 
        verbose=True)

    curr_hparams_df = pd.DataFrame(curr_hparams).T

    # Extract geometry results, add metadata:
    curr_geo_results = curr_results['geo_df']
    geo_meta_cols = list(set(curr_hparams_df.columns) - set(curr_geo_results.columns))
    geo_meta = pd.concat([curr_hparams_df[geo_meta_cols]]*curr_geo_results.shape[0],axis=0)
    geo_meta.index = np.arange(geo_meta.shape[0])
    curr_geo_results['mdl_id'] = mdl_id
    curr_geo_results = pd.concat([curr_geo_results, geo_meta], axis=1)
    geo_results = pd.concat([geo_results, curr_geo_results], axis=0)
    
    # Extract classifier performance results, add metadata:
    curr_perf_results = curr_results['perf_df']
    perf_meta_cols = list(set(curr_hparams_df.columns) - set(curr_perf_results.columns))
    perf_meta = pd.concat([curr_hparams_df[perf_meta_cols]]*curr_perf_results.shape[0],axis=0)
    perf_meta.index = np.arange(perf_meta.shape[0])
    curr_perf_results = pd.concat([curr_perf_results, perf_meta], axis=1)
    curr_perf_results['mdl_id'] = mdl_id
    perf_results = pd.concat([perf_results, curr_perf_results], axis=0)

    # Extract autoencoder representations, add metadata:
    if curr_results['ae_df'] is not None:            
        curr_ae_results = curr_results['ae_df']
        ae_meta_cols = list(set(curr_hparams_df.columns) - set(curr_ae_results.columns))
        ae_meta = pd.concat([curr_hparams_df[ae_meta_cols]]*curr_ae_results.shape[0],axis=0)
        ae_meta.index = np.arange(ae_meta.shape[0])
        curr_ae_results = pd.concat([curr_ae_results, ae_meta], axis=1)
        curr_ae_results['mdl_id'] = mdl_id
        ae_results = pd.concat([ae_results, curr_ae_results])

geo_results['train_partition'] = geo_results.apply(lambda x :str(x.train_partition), axis=1)
stop_mdl = time.time()
print('Duration = {}'.format(stop_mdl - start_mdl))

Fitting autoencoder...
gpu=False
device=cpu
type(clase_train)0=<class 'numpy.ndarray'>
type(clase_train)1=<class 'torch.Tensor'>
0 rec  0.2128680944442749 ce  0.0 sp  0.1469516158103943 total  0.2128680944442749
49 rec  0.003976545296609402 ce  0.0 sp  0.7832909822463989 total  0.003976545296609402
fit_autoencoder duration=303.4644105434418
rename duration=2.384185791015625e-07
layer_cols2rows duration=0.010848045349121094
eliminate learning rows duration=0.002334117889404297
eliminate init/final rows duration=0.003291606903076172
representation_df.shape=(4, 5)
representation_df.iloc[-1].test.shape=(1600, 80)
representation_df.iloc[-1].task0_class_label.shape=(1600,)
row val types =[<class 'str'>, <class 'str'>, <class 'int'>, <class 'numpy.ndarray'>, <class 'numpy.ndarray'>, <class 'numpy.ndarray'>, <class 'numpy.ndarray'>]
sys.getsizeof(curr_rep_ar)=485248
lr_fit duration=0.01716327667236328
lr_fit duration=0.016537189483642578
lr_fit duration=0.013029813766479492
lr_fit duration=0.0

##### Aggregate results:

In [94]:
all_results = dict()
all_results['geo_df'] = geo_results
all_results['perf_df'] = perf_results
all_results['ae_df'] = ae_results

##### Save output:

In [135]:
if sv:
    
    # Save results dataframe:
    curr_output_directory=increment_dir_name(base_output_directory, run_base_name)
    if not os.path.exists(curr_output_directory):
        pathlib.Path(curr_output_directory).mkdir(parents=True, exist_ok=True)
    results_path = os.path.join(curr_output_directory, 'iterate_entangler.pickle')
    print('results_path={}'.format(results_path))
    pickle.dump(all_results, open(results_path, 'wb'))
    
    M = Metadata()
    metadata_consts = find_df_constants(hparams_df)

    # Write task definitions:
    if 'task_defs' in metadata_consts:
        for t, task in enumerate(metadata_consts['task_defs']):
            curr_task_str = ' vs '.join([class_def2str(x) for x in task])
            M.add_param('task{}'.format(t), curr_task_str)

    # Write simulation parameters:
    sim_params_meta = dict()
    for s in simulation_cols:
        if s in metadata_consts:
            sim_params_meta[s] = metadata_consts[s]
    M.add_param('sim_params', sim_params_meta)
    
    # Write model parameters:
    mdl_params_meta = dict()
    for m in autoencoder_cols:
        if m in metadata_consts:
            mdl_params_meta[m] = metadata_consts[m]
    M.add_param('mdl_params', mdl_params_meta)

    M.add_output(results_path)
    M.duration = stop_mdl - start_mdl
    metadata_path = os.path.join(curr_output_directory, 'iterate_entangler_metadata.json')
    write_metadata(M, metadata_path)

results_path=/mnt/smb/locker/issa-locker/users/Dan/code/ws/results/run814/iterate_entangler.pickle
Computing checksum for /mnt/smb/locker/issa-locker/users/Dan/code/ws/results/run814/iterate_entangler.pickle...
type(val_in)=<class 'dict'>
type(val_in)=<class 'list'>
type(val_in)=<class 'int'>
type(val_in)=<class 'numpy.int64'>
type(val_in)=<class 'numpy.float64'>
type(val_in)=<class 'numpy.float64'>
type(val_in)=<class 'str'>
type(val_in)=<class 'numpy.float64'>
type(val_in)=<class 'numpy.int64'>
type(val_in)=<class 'numpy.int64'>
type(val_in)=<class 'numpy.int64'>
type(val_in)=<class 'numpy.float64'>
type(val_in)=<class 'numpy.float64'>
type(val_in)=<class 'numpy.int64'>
type(val_in)=<class 'numpy.int64'>
type(val_in)=<class 'numpy.float64'>
type(val_in)=<class 'numpy.float64'>
type(val_in)=<class 'numpy.int64'>
type(val_in)=<class 'numpy.int64'>
type(val_in)=<class 'numpy.int64'>
type(val_in)=<class 'list'>
type(val_in)=<class 'int'>
type(val_in)=<class 'int'>
type(val_in)=<class 'li